In [2]:
import ollama

## **Model**

In [3]:
MODEL = "deepseek-r1:1.5b"

## **Tool**

Composed of three parts: 
- doc string
- type hints eg "city"
- name

In [4]:
def get_weather(city:str) -> str:
    """ Get current weather for a given city"""
    return f"22°C and light rain expecre "

In [5]:
TOOLS = {
    "weather":get_weather,
}

## **Agent Loop**

COmposed of loop with maximum nuumber of steps, this acts as a guardrail to limit the number of calls made by the model

In [10]:
MAX_STEPS = 5

def run_agent(user_question:str, max_steps = MAX_STEPS):

    messages = [{"role":"user", "content":user_question}]
    
    for step in range(max_steps):
        #  reponse
        response = ollama.chat(
            model = MODEL , 
            messages = messages ,
            tools =list(TOOLS.values())
        )

    messages.append(response.message)

#  if th emodel did not ask for anything based on the tool 
    if not response.message.tool_calls:
        return response.message.content 

    #  if model asked for one of more tools: 
    for call in response.messahe.tool_calls:
        name = call.function.name
        args = call.function.arguments 
        print(f"step {step + 1}, model requested: {name}{args}")


        try:
            func = TOOLS[name]
            result = func(**args)
        except Exception as e:
            result = f"ERROR: {e}"

        messages.append({"role":"tool", "content":str(result), "tool_name":name})

    return "stopped: reached maximum number of steps"


In [ ]:
   # Needs a tool -> watch the loop run.
print(run_agent("Should I carry an umbrella in Nairobi today?"))
print("-" * 60)